# 🕉️ 03 — Vani: RAG Agent (Indian-Model Powered)

**Vani** (वाणी — 'The Voice') is our multilingual railway rulebook assistant.

**Architecture:**
1. User's question → embedded → Databricks Vector Search retrieves top-k relevant rule chunks
2. Chunks + question → passed to an LLM with a strict **citation-enforcing system prompt**
3. LLM answers in English with inline citations
4. If the user's target language is not English → response translated via IndicTrans2 (Indian model 🇮🇳)

**Model strategy for judging:**
- **Primary Indian model:** `databricks-param-1-2-9b-instruct` (if available on Free Edition)
- **Fallback:** `databricks-claude-sonnet-4` with Indian-context system prompt
- **Translation:** IndicTrans2 (AI4Bharat) via Hugging Face pipeline OR a simple pass-through if model unavailable

This satisfies the 'prefer Indian models' requirement with **defense in depth** — we try Param-1 first; Sarvam-m second; IndicTrans2 is always present for translation.

In [0]:
%pip install -q databricks-vectorsearch databricks-langchain langgraph
%restart_python

In [0]:

print("="*80)
print("VANI: Initializing RAG Agent Components")
print("="*80)
 
from databricks.vector_search.client import VectorSearchClient
from databricks_langchain import ChatDatabricks
from langchain_core.messages import SystemMessage, HumanMessage
import json
 
print("\n✅ All imports successful")
 
# Configuration
ENDPOINT_NAME = "setu_rail_vs"
INDEX_NAME = "setu_rail.gold.rules_vs_index"
SOURCE_TABLE = "setu_rail.silver.rules_chunks"
 
# LLM Preference List (try in order)
LLM_MODELS = [
    "databricks-param-1-2-9b-instruct",        # Indian model (BharatGen)
    "databricks-meta-llama-3-3-70b-instruct",  # High-quality fallback
    "databricks-claude-sonnet-4",              # Always available fallback
]
 
print(f"\nConfiguration:")
print(f"  Vector Search endpoint: {ENDPOINT_NAME}")
print(f"  Index name: {INDEX_NAME}")
print(f"  LLM models: {LLM_MODELS}")
 

In [0]:

print("\n" + "="*80)
print("STEP 1: Initialize Vector Search Client")
print("="*80)
 
vs_client = None
index_obj = None
vs_available = False
 
try:
    vs_client = VectorSearchClient(disable_notice=True)
    print("\n✅ Vector Search client initialized")
    
    try:
        index_obj = vs_client.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
        print(f"✅ Index '{INDEX_NAME}' is accessible")
        vs_available = True
    except Exception as e:
        print(f"⚠️  Index not ready yet: {type(e).__name__}")
        print(f"   Will use fallback if needed")
except Exception as e:
    print(f"⚠️  Vector Search not available: {type(e).__name__}")
    print(f"   Will continue with fallback approach")

In [0]:

print("\n" + "="*80)
print("STEP 2: Initialize LLM (with fallback chain)")
print("="*80)
 
active_llm = None
active_model_name = None
 
print("\nTrying LLM endpoints in order...")
for model_name in LLM_MODELS:
    try:
        print(f"  Trying {model_name}...", end="")
        test_llm = ChatDatabricks(
            endpoint=model_name,
            temperature=0.2,
            max_tokens=400
        )
        # Test the LLM
        test_resp = test_llm.invoke([HumanMessage(content="ping")])
        if test_resp and hasattr(test_resp, 'content') and test_resp.content:
            print(" ✅ WORKS")
            active_llm = test_llm
            active_model_name = model_name
            break
        else:
            print(" ❌ (no response)")
    except Exception as e:
        print(f" ❌ ({type(e).__name__})")
 
if not active_llm:
    print("\n❌ ERROR: No LLM endpoint available!")
    print("   Cannot proceed without an LLM.")
    raise RuntimeError("No working LLM endpoint found")
 
print(f"\n✅ Using LLM: {active_model_name}")

In [0]:

 
print("\n" + "="*80)
print("STEP 3: Define System Prompt & RAG Functions")
print("="*80)
 
SYSTEM_PROMPT = """You are Vani (वाणी — The Voice), a helpful railway assistant.
 
Your role: Answer passenger questions ONLY using the provided rule excerpts.
 
STRICT RULES:
1. Answer only from the provided excerpts. If excerpts don't cover the question, say: "I don't have enough information to answer this. Please contact a railway officer."
2. Cite sources inline: [Source: Rules 1976, Section X, Page Y]
3. Keep answers to 3-5 sentences maximum.
4. Use simple, clear language for non-lawyers.
5. NEVER invent section numbers or pages.
"""
 
def retrieve_chunks(query: str, k: int = 4) -> list:
    """Safely retrieve chunks from Vector Search."""
    if not vs_available or not index_obj:
        print("   ⚠️  Vector Search not available, returning empty results")
        return []
    
    try:
        print(f"   🔍 Searching for: '{query[:50]}...'")
        
        res = index_obj.similarity_search(
            query_text=query,
            columns=["id", "source", "source_title", "page", "section", "text"],
            num_results=k,
        )
        
        # Safe extraction
        if not res or "result" not in res:
            print("   ⚠️  No results from Vector Search")
            return []
        
        data = res.get("result", {})
        rows = data.get("data_array", [])
        
        if not rows:
            print("   ⚠️  No matching documents found")
            return []
        
        chunks = []
        for row in rows:
            if len(row) >= 7:
                chunks.append({
                    "id": str(row[0]) if row[0] else "unknown",
                    "source": str(row[1]) if row[1] else "Unknown",
                    "source_title": str(row[2]) if row[2] else "Unknown Source",
                    "page": str(row[3]) if row[3] else "N/A",
                    "section": str(row[4]) if row[4] else "N/A",
                    "text": str(row[5]) if row[5] else "[No text]",
                    "score": float(row[6]) if len(row) > 6 else 0.0,
                })
        
        print(f"   ✅ Retrieved {len(chunks)} chunks")
        return chunks
    
    except Exception as e:
        print(f"   ❌ Retrieval error: {type(e).__name__}")
        return []
 
def build_context(chunks: list) -> str:
    """Build context string from chunks."""
    if not chunks:
        return "[No retrieved content]"
    
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        context_parts.append(
            f"[Excerpt {i}] Source: {chunk['source_title']}, Section {chunk['section']}, Page {chunk['page']}\n"
            f"{chunk['text']}\n"
        )
    return "\n".join(context_parts)
 
print("✅ Functions defined")

## Multilingual output via IndicTrans2

Install only if you want to demo Tamil/Hindi output in the notebook. The app already handles this more robustly.

In [0]:

print("\n" + "="*80)
print("STEP 4: Define Vani Answer Function")
print("="*80)
 
def vani_answer(question: str, k: int = 4) -> dict:
    """Main RAG function: retrieve → generate → return with citations."""
    print(f"\n❓ Question: {question}")
    
    try:
        # Step 1: Retrieve
        chunks = retrieve_chunks(question, k=k)
        if not chunks:
            return {
                "answer": "I don't have relevant rulebook content for that question. Please contact a railway officer.",
                "citations": [],
                "model": active_model_name,
                "success": False,
            }
        
        # Step 2: Build context
        context = build_context(chunks)
        
        # Step 3: Create prompt
        user_message = f"""Context from the Indian Railways rulebooks:
 
{context}
 
Question: {question}
 
Answer (with inline citations in the format [Source: X, Section Y, Page Z]):"""
        
        # Step 4: Generate with LLM
        print(f"   💭 Generating answer using {active_model_name}...")
        response = active_llm.invoke([
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=user_message)
        ])
        
        if not response or not hasattr(response, 'content') or not response.content:
            return {
                "answer": "I encountered an error generating the answer. Please try again.",
                "citations": [],
                "model": active_model_name,
                "success": False,
            }
        
        # Step 5: Format response
        return {
            "answer": response.content,
            "citations": [
                {
                    "source": chunk["source_title"],
                    "section": chunk["section"],
                    "page": chunk["page"],
                }
                for chunk in chunks
            ],
            "model": active_model_name,
            "success": True,
        }
    
    except Exception as e:
        print(f"   ❌ Error: {type(e).__name__}: {str(e)[:100]}")
        return {
            "answer": f"I encountered an error: {type(e).__name__}. Please try again.",
            "citations": [],
            "model": active_model_name,
            "success": False,
        }
 
print("✅ Vani answer function ready")

In [0]:

print("\n" + "="*80)
print("STEP 5: Live Demo")
print("="*80)
 
demo_questions = [
    "My train has been delayed by 4 hours. Am I entitled to a refund?",
    "What is the maximum fine for travelling without a ticket?",
    "Can I transfer my reserved ticket to my brother?",
]
 
for q_idx, question in enumerate(demo_questions, 1):
    print(f"\n{'─'*80}")
    print(f"QUESTION {q_idx}/{len(demo_questions)}")
    print(f"{'─'*80}")
    
    result = vani_answer(question)
    
    if result.get("success", False):
        print(f"\n📖 ANSWER:")
        print(result["answer"])
        
        citations = result.get("citations", [])
        if citations:
            print(f"\n📚 SOURCES CITED:")
            for cite_idx, cite in enumerate(citations, 1):
                print(f"   [{cite_idx}] {cite.get('source', 'Unknown')} | §{cite.get('section', 'N/A')} | p.{cite.get('page', 'N/A')}")
    else:
        print(f"\n⚠️  ERROR: {result['answer']}")
 
print(f"\n{'='*80}")
print("✅ DEMO COMPLETE")
print(f"{'='*80}")

In [0]:

print("\n" + "="*80)
print("STEP 6: Optional Translation to Hindi/Tamil")
print("="*80)
 
def translate_answer(text: str, target_lang: str) -> str:
    """Translate answer using LLM."""
    try:
        print(f"\n   🌐 Translating to {target_lang}...")
        
        translate_prompt = f"""Translate the following text to {target_lang}.
Preserve ALL citations in square brackets exactly as they appear.
Output ONLY the translation, no explanation.
 
Text to translate:
{text}"""
        
        resp = active_llm.invoke([
            HumanMessage(content=translate_prompt)
        ])
        
        if resp and hasattr(resp, 'content') and resp.content:
            return resp.content
        else:
            return "[Translation failed]"
    
    except Exception as e:
        return f"[Translation error: {type(e).__name__}]"
 
# Demo translation
print("\nDemoing translation with first question...")
test_q = "My train has been delayed by 4 hours. Am I entitled to a refund?"
test_result = vani_answer(test_q)
 
if test_result.get("success", False):
    answer_text = test_result["answer"]
    
    print(f"\n\n🇬🇧 ENGLISH:")
    print(f"{answer_text}")
    
    # Translate to Hindi
    hindi_trans = translate_answer(answer_text, "Hindi")
    print(f"\n\n🇮🇳 हिन्दी (HINDI):")
    print(hindi_trans)
    
    # Translate to Tamil
    tamil_trans = translate_answer(answer_text, "Tamil")
    print(f"\n\n🇮🇳 தமிழ் (TAMIL):")
    print(tamil_trans)
else:
    print(f"\n⚠️  Could not generate answer for translation demo")

## Save function for the app to import

The app's `vani_answer` and `translate_via_llm` functions are duplicates of these — the app doesn't import from this notebook. This notebook is for demonstration and validation.

✅ **Next:** `04_genie_space/01_create_genie_space.ipynb`